# Experimento — Perceptron multicamadas (MLP)

Placeholder do experimento. O texto abaixo é o roteiro. Não há treino neste arquivo.

Rede treinada do zero para classificar `Churn`. Serve de leitura direta das frentes de rede neural: tudo o que está listado abaixo existe como controle deste modelo.


## Como o experimento avança

O experimento é uma sequência de blocos. Cada bloco faz uma intervenção só: um hiperparâmetro, uma escolha de arquitetura ou uma mudança nos dados. O bloco seguinte parte da configuração que venceu, e só fica no lugar dela se a métrica de validação subir.

A métrica é escolhida uma vez, antes do bloco 0 de qualquer modelo, e vale para os sete notebooks. Cerca de 26,5% dos clientes cancelam, então a métrica tem de enxergar a classe de churn. O conjunto de teste não entra nessa escolha. Ele só é usado no bloco final, com a configuração já congelada.

Empate fica com a configuração mais simples. Cada bloco aceito registra o que mudou, a configuração completa e a média e a dispersão nas cinco dobras.

O gradient boosting deste repositório é a referência de comparação. Cada modelo anota a distância do seu melhor bloco até o boosting congelado com o mesmo protocolo. O boosting também anda por blocos; a comparação usa a versão já aceita, sem perseguir um número que ainda está mudando.


## Dados

A entrada é a saída de `if702-preprocessamento.ipynb`: as cinco dobras estratificadas e a função `preparar` para o treino final e o teste. Em cada dobra o scaler é ajustado só no treino da dobra.

O oversampling repetitivo já está feito aí. O `RandomOverSampler` só duplica o treino da dobra. A validação e o teste continuam com um cliente por linha. Nenhum bloco refaz esse balanceamento por fora das dobras.

### Estratégias de dados compartilhadas

Nenhuma, além do pré-processamento.

Quando aparecer uma estratégia nova de exploração ou de mineração, ela é escrita nesta seção e copiada para os outros seis notebooks. Vira um bloco em cada experimento, em cima da configuração que aquele modelo já tiver aceito. O bloco mexe só nos dados. O modelo fica parado.


## Frentes já definidas

Camadas e unidades intermediárias entram juntas, no formato de uma lista de larguras. O bloco de capacidade altera uma coisa: ou a profundidade, ou a largura de uma camada.

A função de ativação das camadas ocultas percorre logística, tangente hiperbólica e ReLU. A saída da classificação fica na escala da probabilidade, independente dessa escolha.

O algoritmo de aprendizagem é o método que atualiza os pesos. As opções são Adam, SGD e RMSprop. Adam e RMSprop já são taxa de aprendizagem adaptativa: Adam combina a média e a variação recente do gradiente; RMSprop ajusta o passo pela magnitude recente do gradiente. O bloco 0 começa em Adam. SGD entra com momento, porque em algumas tarefas essa combinação fica mais estável do que Adam. L-BFGS continua na comparação, para a base pequena. A variação da taxa de aprendizagem percorre os valores iniciais. Se o algoritmo aceito for o SGD, um bloco seguinte ainda pode deixar a taxa constante, fazê-la cair ao longo do tempo ou reduzi-la quando a validação estaciona. Com Adam ou RMSprop esse bloco não se repete.

O tamanho do lote é uma frente própria. Ele define quantos clientes entram em cada atualização dos pesos.

O número de épocas é o número de passagens completas pelo treino da dobra. Épocas de menos deixam a rede subajustada; épocas demais a deixam decorar o treino. Esse número não é escolhido no escuro: o treino para por early stopping na validação da dobra, e a paciência desse critério é o hiperparâmetro a mais. A paciência é quantas épocas a validação pode passar sem melhorar antes da parada. O número de épocas aceito é o que a validação escolheu dentro dessa paciência.

Dropout e regularização L2 (weight decay) são blocos próprios. O dropout zera neurônios ao acaso durante o treino; as taxas percorridas ficam entre 0,2 e 0,5, e ele pesa mais quando a rede já está funda e larga. A L2 soma à função de erro uma penalidade sobre pesos grandes, e cada bloco testa uma força dessa penalidade, separado do dropout. A implementação precisa expor dropout; o `MLPClassifier` do scikit-learn não tem esse controle, então a rede fica numa implementação que tenha.

O oversampling repetitivo é o do pré-processamento, compartilhado com os outros experimentos.


## Hiperparâmetros que ainda vão chegar

Camadas, unidades, ativação, taxa, lote, épocas com early stopping, paciência, dropout, L2 e otimizador já estão nas frentes acima. O que ainda chegar entra como bloco novo, um item de cada vez, a partir da configuração vencedora.


## Ordem prevista dos blocos

1. Bloco 0. Uma rede pequena, ReLU, Adam, um lote e uma paciência iniciais, sem dropout e com regularização L2 fraca. O early stopping já vale, para o número de épocas sair da validação. Só estabelece o número.
2. Capacidade. Profundidade, depois largura.
3. Ativação. Logística, tangente hiperbólica e ReLU, na largura vencedora.
4. Taxa de aprendizagem. Em seguida o otimizador: Adam, RMSprop, SGD com momento e L-BFGS. A taxa adaptativa do SGD só entra se o SGD tiver vencido.
5. Tamanho do lote.
6. Paciência do early stopping. O número de épocas fica o que a validação escolher dentro dessa paciência.
7. Dropout, nas taxas de 0,2 a 0,5, e depois a força da L2. Um de cada vez.
8. Itens que ainda chegarem.
9. Estratégias de dados compartilhadas, na ordem em que forem propostas.
10. Bloco final. A configuração aceita é retreinada com `preparar` e avaliada no teste.


## O que este arquivo ainda não contém

Não há código, treino, busca nem número medido. Este notebook é o roteiro. O primeiro bloco só começa depois que a métrica comum estiver escrita.
